# Export IL S2 and S5 Rasters

## About this Notebook

### Objective

Generate a consolidated Sentinel-2 (S2) and Sentinel-5P (S5) dataset for Illinois facilities and/or sensors.

This notebook extracts, aligns, and exports both high-resolution optical imagery (S2) and atmospheric NO₂ measurements (S5) for a defined Illinois study region, preparing data for multimodal modeling and regional analysis.

---

### Inputs

- Cleaned ground sensor or facility dataset (Illinois subset)
- Sentinel-2 surface reflectance imagery (12 bands)
- Sentinel-5P TROPOMI NO₂ data
- Geographic boundaries for Illinois (or Illinois facility filter)
- Temporal aggregation settings (e.g., quarterly or annual)

Primary data bucket:
`gs://msads-mba-capstone-team-1/Data/`

---

### Outputs

- Sentinel-2 image chips for Illinois locations
- Sentinel-5P NO₂ raster patches or aggregated features
- Spatially aligned multimodal dataset
- Exported files saved to GCS

Example output directories:
- `gs://msads-mba-capstone-team-1/Data/IL_S2_images/`
- `gs://msads-mba-capstone-team-1/Data/IL_S5_images/`

These outputs serve as the Illinois regional dataset for modeling and environmental justice analysis.

---

### Workflow Overview

1. Filter facilities/sensors to Illinois  
2. Loop over locations and time periods  
3. Extract Sentinel-2 image chips  
4. Extract or aggregate Sentinel-5P NO₂ values  
5. Validate spatial and temporal alignment  
6. Export standardized multimodal outputs to GCS  

---

### Key Notes

- Illinois filtering must occur before satellite extraction to reduce compute costs.
- Sentinel-2 and Sentinel-5P spatial resolutions differ; buffer sizes must reflect this.
- Temporal aggregation must match modeling targets.
- CRS alignment must be verified before clipping.
- Outputs must remain schema-consistent with national pipeline notebooks.

---

### Pipeline Position

ExportSensorData → ExportILS2S5 → FusionNO2Model (Illinois Prototype)

This notebook supports regional prototyping and environmental justice analysis within Illinois.

## Config

In [4]:
ee_project_name = "msads-mba-autumn-2025-team-1"
bucket_name = "msads-mba-capstone-team-1"
gcs_folder_s2 = "Data/TrainingData/S2_Statewide"
gcs_folder_s5 = "Data/TrainingData/S5_Statewide"

## Export S2 Statewide Raster

In [2]:
import ee
import time
import datetime

ee.Initialize(project=ee_project_name)

target_year = 2021
QUARTERS = ["Q1", "Q2", "Q3", "Q4"]

S2_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"

S2_BANDS = [
    "B1", "B2", "B3", "B4",
    "B5", "B6", "B7", "B8",
    "B8A", "B9", "B11", "B12"
]

In [11]:
# Get Illinois boundary
states = ee.FeatureCollection("TIGER/2018/States")
illinois = states.filter(ee.Filter.eq("NAME", "Illinois")).geometry()

In [12]:
def resample_to_10m(image):
    return image.resample("bilinear").reproject(
        crs="EPSG:32616",
        scale=10
    )

In [13]:
def get_quarterly_s2_image(start_date, end_date):
    
    full_collection = (
        ee.ImageCollection(S2_COLLECTION)
        .filterBounds(illinois)
        .filterDate(start_date, end_date)
    )
    
    filtered_collection = full_collection.filter(
        ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)
    )
    
    filtered_size = filtered_collection.size()
    full_size = full_collection.size()
    
    image = ee.Algorithms.If(
        filtered_size.gt(0),
        filtered_collection.sort("CLOUDY_PIXEL_PERCENTAGE").first(),
        ee.Algorithms.If(
            full_size.gt(0),
            full_collection.median(),
            None
        )
    )
    
    return ee.Image(image)

In [14]:
for quarter in QUARTERS:
    
    print("\n==============================================")
    print(f"[{datetime.datetime.now()}]")
    print(f"Building statewide S2 composite for {target_year} {quarter}")
    print("==============================================")
    
    # Define dates
    if quarter == "Q1":
        start_date = f"{target_year}-01-01"
        end_date   = f"{target_year}-03-31"
    elif quarter == "Q2":
        start_date = f"{target_year}-04-01"
        end_date   = f"{target_year}-06-30"
    elif quarter == "Q3":
        start_date = f"{target_year}-07-01"
        end_date   = f"{target_year}-09-30"
    elif quarter == "Q4":
        start_date = f"{target_year}-10-01"
        end_date   = f"{target_year}-12-31"
    
    print(f"Date range: {start_date} → {end_date}")
    
    # Build composite
    image = get_quarterly_s2_image(start_date, end_date)
    
    if image is None:
        print("No imagery available.")
        continue
    
    image = image.select(S2_BANDS)
    image = resample_to_10m(image)
    
    file_name = f"S2_Statewide_{target_year}{quarter}"
    
    print("Submitting export task...")
    
    task = ee.batch.Export.image.toCloudStorage(
        image=image.clip(illinois),
        description=file_name,
        bucket=bucket_name,
        fileNamePrefix=f"{gcs_folder_s2}/{file_name}",
        region=illinois,
        scale=10,                  # LOCK 10m resolution
        crs="EPSG:32616",          # LOCK projection
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )
    
    task.start()
    
    print(f"Export started: {file_name}")

print("\nAll quarterly statewide exports submitted.")


[2026-02-20 16:39:24.895898]
Building statewide S2 composite for 2021 Q1
Date range: 2021-01-01 → 2021-03-31
Submitting export task...
Export started: S2_Statewide_2021Q1

[2026-02-20 16:39:25.188479]
Building statewide S2 composite for 2021 Q2
Date range: 2021-04-01 → 2021-06-30
Submitting export task...
Export started: S2_Statewide_2021Q2

[2026-02-20 16:39:25.479444]
Building statewide S2 composite for 2021 Q3
Date range: 2021-07-01 → 2021-09-30
Submitting export task...
Export started: S2_Statewide_2021Q3

[2026-02-20 16:39:25.772706]
Building statewide S2 composite for 2021 Q4
Date range: 2021-10-01 → 2021-12-31
Submitting export task...
Export started: S2_Statewide_2021Q4

All quarterly statewide exports submitted.


## Export S5 Statewide Raster

In [3]:
import ee
import datetime

ee.Initialize(project=ee_project_name)

target_year = 2021
QUARTERS = ["Q1", "Q2", "Q3", "Q4"]

S5_COLLECTION = "COPERNICUS/S5P/OFFL/L3_NO2"
S5_BAND = "tropospheric_NO2_column_number_density"

In [16]:
states = ee.FeatureCollection("TIGER/2018/States")
illinois = states.filter(ee.Filter.eq("NAME", "Illinois")).geometry()

In [17]:
def upscale_to_10m(image):
    return image.resample("bilinear").reproject(
        crs="EPSG:32616",
        scale=10
    )

In [18]:
def get_quarterly_s5_image(start_date, end_date):
    
    collection = (
        ee.ImageCollection(S5_COLLECTION)
        .filterBounds(illinois)
        .filterDate(start_date, end_date)
        .select(S5_BAND)
    )
    
    size = collection.size()
    
    image = ee.Algorithms.If(
        size.gt(0),
        collection.mean(),
        None
    )
    
    return ee.Image(image)

In [19]:
for quarter in QUARTERS:
    
    print("\n==============================================")
    print(f"[{datetime.datetime.now()}]")
    print(f"Building statewide S5 composite for {target_year} {quarter}")
    print("==============================================")
    
    # Define dates
    if quarter == "Q1":
        start_date = f"{target_year}-01-01"
        end_date   = f"{target_year}-03-31"
    elif quarter == "Q2":
        start_date = f"{target_year}-04-01"
        end_date   = f"{target_year}-06-30"
    elif quarter == "Q3":
        start_date = f"{target_year}-07-01"
        end_date   = f"{target_year}-09-30"
    elif quarter == "Q4":
        start_date = f"{target_year}-10-01"
        end_date   = f"{target_year}-12-31"
    
    print(f"Date range: {start_date} → {end_date}")
    
    image = get_quarterly_s5_image(start_date, end_date)
    
    if image is None:
        print("No S5 imagery available.")
        continue
    
    # Match S2 spatial grid
    image = upscale_to_10m(image)
    
    file_name = f"S5_Statewide_{target_year}{quarter}"
    
    print("Submitting export task...")
    
    task = ee.batch.Export.image.toCloudStorage(
        image=image.clip(illinois),
        description=file_name,
        bucket=bucket_name,
        fileNamePrefix=f"{gcs_folder_s5}/{file_name}",
        region=illinois,
        scale=10,                  # Force 10m grid
        crs="EPSG:32616",          # Match S2
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )
    
    task.start()
    
    print(f"Export started: {file_name}")

print("\nAll statewide S5 exports submitted.")


[2026-02-20 16:58:54.173273]
Building statewide S5 composite for 2021 Q1
Date range: 2021-01-01 → 2021-03-31
Submitting export task...
Export started: S5_Statewide_2021_Q1

[2026-02-20 16:58:54.448012]
Building statewide S5 composite for 2021 Q2
Date range: 2021-04-01 → 2021-06-30
Submitting export task...
Export started: S5_Statewide_2021_Q2

[2026-02-20 16:58:54.669133]
Building statewide S5 composite for 2021 Q3
Date range: 2021-07-01 → 2021-09-30
Submitting export task...
Export started: S5_Statewide_2021_Q3

[2026-02-20 16:58:54.972366]
Building statewide S5 composite for 2021 Q4
Date range: 2021-10-01 → 2021-12-31
Submitting export task...
Export started: S5_Statewide_2021_Q4

All statewide S5 exports submitted.


In [5]:
import pandas as pd
import numpy as np

# Load NEI 2021
nei = pd.read_csv(
    "gs://msads-mba-capstone-team-1/Data/TrainingData/nei_2021_IL_nox_for_model.csv"
)

print("Shape:", nei.shape)
print("\nColumns:\n", nei.columns)
print("\nMissing values:\n", nei.isnull().sum().sort_values(ascending=False).head(15))

Shape: (2319, 11)

Columns:
 Index(['eis facility id', 'company name', 'site name', 'primary naics code',
       'primary naics description', 'facility source type', 'site latitude',
       'site longitude', 'GEOID10', 'total emissions', 'log_emissions'],
      dtype='object')

Missing values:
 facility source type         1062
company name                  751
eis facility id                 0
site name                       0
primary naics code              0
primary naics description       0
site latitude                   0
site longitude                  0
GEOID10                         0
total emissions                 0
log_emissions                   0
dtype: int64


In [8]:
# ============================================
# FACILITY FILTER CONFIG
# ============================================

start_index = None        # e.g. 0
end_index = None          # e.g. 100
target_facility_ids = None  # e.g. ["12345", "67890"]

# --------------------------------------------
# Apply filtering
# --------------------------------------------

df_to_export = nei.copy()

# Filter by specific facility IDs
if target_facility_ids is not None:
    target_facility_ids = set(str(fid) for fid in target_facility_ids)
    df_to_export = df_to_export[
        df_to_export["eis facility id"].astype(str).isin(target_facility_ids)
    ]

# Apply index slicing AFTER ID filtering
if start_index is not None and end_index is not None:
    df_to_export = df_to_export.iloc[start_index:end_index]

df_to_export = df_to_export.reset_index(drop=True)

print("====================================")
print("Total facilities in full dataset:", len(nei))
print("Facilities selected for export:", len(df_to_export))
print("====================================")

Total facilities in full dataset: 2319
Facilities selected for export: 2319


In [9]:
import datetime

total_facilities = len(df_to_export)

for q_idx, quarter in enumerate(QUARTERS, 1):
    
    quarter_start_time = time.time()
    print("\n====================================================")
    print(f"[{datetime.datetime.now()}]")
    print(f"Starting Quarter {q_idx}/{len(QUARTERS)}: {target_year} {quarter}")
    print("====================================================")
    
    # ---------------------------------------------
    # Define quarterly dates
    # ---------------------------------------------
    if quarter == "Q1":
        start_date = f"{target_year}-01-01"
        end_date   = f"{target_year}-03-31"
    elif quarter == "Q2":
        start_date = f"{target_year}-04-01"
        end_date   = f"{target_year}-06-30"
    elif quarter == "Q3":
        start_date = f"{target_year}-07-01"
        end_date   = f"{target_year}-09-30"
    elif quarter == "Q4":
        start_date = f"{target_year}-10-01"
        end_date   = f"{target_year}-12-31"
    
    print(f"Date Range: {start_date} → {end_date}")
    print(f"Facilities to process this run: {total_facilities}")
    print("Building quarterly composite...")
    
    # ---------------------------------------------
    # BUILD IMAGE ONCE
    # ---------------------------------------------
    quarterly_image = get_quarterly_s2_image(start_date, end_date)
    
    if quarterly_image is None:
        print("No imagery available for this quarter.")
        continue
    
    quarterly_image = quarterly_image.select(S2_BANDS)
    quarterly_image = resample_to_10m(quarterly_image)
    
    print("Quarterly composite ready.")
    print("Starting facility exports...\n")
    
    # ---------------------------------------------
    # Loop filtered facilities
    # ---------------------------------------------
    for i, row in df_to_export.iterrows():
        
        facility_id = str(row["eis facility id"])
        lat = float(row["site latitude"])
        lon = float(row["site longitude"])
        
        progress = i + 1
        
        print("--------------------------------------------------")
        print(f"[{datetime.datetime.now()}]")
        print(f"Quarter: {quarter}")
        print(f"Facility {progress}/{total_facilities}")
        print(f"Facility ID: {facility_id}")
        print(f"Lat/Lon: ({lat:.6f}, {lon:.6f})")
        
        chip, region = create_chip(quarterly_image, lat, lon)
        
        file_name = f"S2_Facility{facility_id}_{target_year}_{quarter}"
        
        wait_for_available_slot(max_active_tasks)
        
        active_tasks = len([
            t for t in ee.batch.Task.list()
            if t.status()["state"] in ["READY", "RUNNING"]
        ])
        
        print(f"Active EE tasks: {active_tasks}")
        print("Submitting export task...")
        
        task = ee.batch.Export.image.toCloudStorage(
            image=chip,
            description=file_name,
            bucket=bucket_name,
            fileNamePrefix=f"{gcs_folder}/{file_name}",
            region=region,
            dimensions="120x120",
            fileFormat="GeoTIFF",
            maxPixels=1e13
        )
        
        task.start()
        
        print(f"Export started: {file_name}")
    
    quarter_time = time.time() - quarter_start_time
    
    print("\n====================================================")
    print(f"Finished Quarter {quarter}")
    print(f"Time taken: {quarter_time/60:.2f} minutes")
    print("====================================================\n")

print("\nAll export tasks submitted.")


[2026-02-20 16:27:36.490810]
Starting Quarter 1/4: 2021 Q1
Date Range: 2021-01-01 → 2021-03-31
Facilities to process this run: 2319
Building quarterly composite...
Quarterly composite ready.
Starting facility exports...

--------------------------------------------------
[2026-02-20 16:27:36.493169]
Quarter: Q1
Facility 1/2319
Facility ID: 558811
Lat/Lon: (40.284400, -88.415300)
Active EE tasks: 0
Submitting export task...
Export started: S2_Facility558811_2021_Q1
--------------------------------------------------
[2026-02-20 16:31:50.638391]
Quarter: Q1
Facility 2/2319
Facility ID: 558911
Lat/Lon: (41.772942, -87.801655)
Active EE tasks: 1
Submitting export task...
Export started: S2_Facility558911_2021_Q1
--------------------------------------------------
[2026-02-20 16:35:57.097793]
Quarter: Q1
Facility 3/2319
Facility ID: 559311
Lat/Lon: (41.784732, -87.818156)


KeyboardInterrupt: 